# GenAI Simulation Engine — Quickstart Notebook

This notebook walks through:
1. Training a small DDPM model on CIFAR-10
2. Generating synthetic samples
3. Computing FID/IS evaluation metrics
4. Dataset augmentation workflow

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

## 1. Build and inspect the UNet backbone

In [ ]:
from src.models.unet import build_unet

model = build_unet('small', in_channels=3, out_channels=3, num_classes=10)
n_params = sum(p.numel() for p in model.parameters())
print(f'UNet (small): {n_params:,} parameters ({n_params/1e6:.1f}M)')

# Test forward pass
x = torch.randn(4, 3, 32, 32)
t = torch.randint(0, 1000, (4,))
labels = torch.randint(0, 10, (4,))
out = model(x, t, labels)
print(f'Input: {x.shape} → Output: {out.shape}')

## 2. Visualize the noise schedule

In [ ]:
from src.models.noise_scheduler import DDPMScheduler

sched_cosine = DDPMScheduler(1000, beta_schedule='cosine')
sched_linear = DDPMScheduler(1000, beta_schedule='linear')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sched_linear.betas.numpy(),  label='Linear',  color='#ef4444')
axes[0].plot(sched_cosine.betas.numpy(),  label='Cosine',  color='#7c3aed')
axes[0].set_title('Beta Schedule'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(sched_linear.alphas_cumprod.numpy(), label='Linear',  color='#ef4444')
axes[1].plot(sched_cosine.alphas_cumprod.numpy(), label='Cosine',  color='#7c3aed')
axes[1].set_title('Cumulative Alpha (signal retention)'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Forward diffusion — visualize noising process

In [ ]:
from torchvision.datasets import CIFAR10
import torchvision.transforms as T

tf = T.Compose([T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
ds = CIFAR10('data/cifar10', download=True, transform=tf)
x0 = ds[0][0].unsqueeze(0)  # [1, 3, 32, 32]

sched = DDPMScheduler(1000, 'cosine')
timesteps_to_show = [0, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(14, 3))
for ax, t in zip(axes, timesteps_to_show):
    t_tensor = torch.tensor([t])
    noisy = sched.add_noise(x0, timesteps=t_tensor).noisy_sample
    img = ((noisy[0].permute(1,2,0).numpy() + 1) / 2).clip(0,1)
    ax.imshow(img)
    ax.set_title(f't={t}', fontsize=9)
    ax.axis('off')
plt.suptitle('Forward Diffusion Process (q(x_t | x_0))', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Quick training loop (toy example, 50 steps)

In [ ]:
import torch.nn.functional as F
from torch.optim import AdamW

model = build_unet('small').to(device)
optimizer = AdamW(model.parameters(), lr=2e-4)
sched = DDPMScheduler(100, 'cosine')  # only 100 timesteps for demo

losses = []
for step in range(50):
    x = torch.randn(8, 3, 32, 32).to(device)  # fake data
    noise = torch.randn_like(x)
    t = torch.randint(0, 100, (8,)).to(device)
    noisy = sched.add_noise(x, noise, t).noisy_sample
    pred = model(noisy, t)
    loss = F.mse_loss(pred, noise)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 10 == 0:
        print(f'Step {step:3d} | loss={loss.item():.4f}')

plt.figure(figsize=(8, 3))
plt.plot(losses, color='#7c3aed', linewidth=2)
plt.title('Training Loss'); plt.xlabel('Step'); plt.ylabel('MSE Loss')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. EMA demonstration

In [ ]:
from src.utils.ema import EMAModel

model = build_unet('small')
ema = EMAModel(model, decay=0.9999)

# Simulate 100 update steps
for _ in range(100):
    # Simulate a gradient update
    with torch.no_grad():
        for p in model.parameters():
            p.add_(torch.randn_like(p) * 0.01)
    ema.update(model)

print('EMA decay (after warmup):', ema.get_decay())
print('Shadow params tracked:', len(ema.shadow_params))

# Apply EMA for inference
ema.apply_shadow(model)
print('EMA weights applied for inference')
ema.restore(model)
print('Original weights restored')